# 08 — Patient-cluster bootstrap intervals and paired tests

This notebook computes nothing until Notebook 07 reports `ready: true`. It verifies the upstream file checksum, record count, frozen cohort fingerprint, four-condition coverage, and identical paired query sets before statistical analysis.

Label-derived outcomes are recomputed in every patient/group bootstrap and permutation replicate from cluster-level event counts. The vectorized engine implements the same patient/source-cluster resampling and swapping protocol without reconstructing a DataFrame for each replicate. Reference vectors and patient/source cluster assignments must agree across paired arms. Raw p-values are adjusted with Holm correction within each prespecified model-specific strategy-comparison family. Between-model comparisons use a separate family within each metric, bundle, source dataset, and condition.

In [ ]:
from pathlib import Path
import json, os, sys

def find_rerun_dir():
    candidates = [
        Path(os.environ.get("JAMIA_RERUN_DIR", "")),
        Path.cwd(),
        Path.cwd().parent,
    ]
    for candidate in candidates:
        if str(candidate) and (candidate / "rerun_config.json").exists():
            return candidate.resolve()
    raise FileNotFoundError("Set JAMIA_RERUN_DIR to the folder containing rerun_config.json")

RERUN_DIR = find_rerun_dir()
# Never expose the implementation directory as a top-level import location:
# rerun_code/statistics.py would shadow Python's standard-library statistics.
implementation_dir = (RERUN_DIR / "src" / "rerun_code").resolve()
clean_sys_path = []
for entry in sys.path:
    try:
        resolved_entry = Path(entry or ".").resolve()
    except Exception:
        resolved_entry = None
    if resolved_entry != implementation_dir:
        clean_sys_path.append(entry)
sys.path[:] = clean_sys_path
sys.path.insert(0, str(RERUN_DIR / "src"))
from rerun_code.config import load_config, output_paths
RERUN_DIR, CONFIG = load_config(RERUN_DIR)
PATHS = output_paths(CONFIG)
print("Code:", RERUN_DIR)
print("Output:", PATHS["root"])

In [ ]:
import importlib, itertools, pandas as pd
from rerun_code.common import read_jsonl, write_json
from rerun_code.config import sha256_path
from rerun_code.generation import CONDITIONS
from rerun_code.report_labeler import _cohort_fingerprint
import rerun_code.statistics as statistics_module
statistics_module = importlib.reload(statistics_module)
required_statistics_api = 3
actual_statistics_api = int(getattr(statistics_module, "STATISTICS_API_VERSION", 0))
if actual_statistics_api < required_statistics_api:
    raise ImportError(
        "Notebook 08 and rerun_code/statistics.py are out of sync. "
        f"Required statistics API {required_statistics_api}, found {actual_statistics_api} at "
        f"{Path(statistics_module.__file__).resolve()}. Copy the updated statistics.py, "
        "restart the kernel, and rerun this cell."
    )
from rerun_code.statistics import cluster_bootstrap, paired_cluster_permutation, holm_adjust

upstream_status_path = PATHS["metrics"] / "notebook07_status.json"
status_path = PATHS["statistics"] / "notebook08_status.json"
statistics_ready = False
pending_reason = None
upstream = {}
if not upstream_status_path.exists():
    pending_reason = "Notebook 07 has not written notebook07_status.json."
else:
    upstream = json.loads(upstream_status_path.read_text(encoding="utf-8"))
    if not upstream.get("ready"):
        pending_reason = str(
            upstream.get("pending_reason") or "Notebook 07 metrics are not ready."
        )

write_json(status_path, {
    "ready": False,
    "upstream_status": str(upstream_status_path),
    "pending_reason": pending_reason or "Statistical computation is in progress.",
})

if pending_reason:
    print("NOTEBOOK 07 OUTPUTS ARE PENDING — NOTEBOOK 08 DID NOT COMPUTE STATISTICS.")
    print("Reason:", pending_reason)
    print("Complete human validation in Notebook 06, then run Notebook 07.")
else:
    per_study_path = Path(upstream.get("per_study_metrics", ""))
    expected_sha256 = str(upstream.get("per_study_metrics_sha256", ""))
    if not per_study_path.exists():
        raise FileNotFoundError(f"Notebook 07 per-study metrics are missing: {per_study_path}")
    actual_sha256 = sha256_path(per_study_path)
    if not expected_sha256 or actual_sha256 != expected_sha256:
        raise AssertionError(
            "Notebook 07 per-study metrics checksum differs from notebook07_status.json"
        )
    frame = pd.DataFrame(read_jsonl(per_study_path))
    if frame.empty or len(frame) != int(upstream.get("n_records", -1)):
        raise AssertionError("Notebook 07 per-study record count is missing or inconsistent")
    if frame["generation_record_id"].astype(str).duplicated().any():
        raise AssertionError("Notebook 07 per-study metrics contain duplicate identifiers")
    actual_fingerprint = _cohort_fingerprint(frame)
    if actual_fingerprint != upstream.get("generation_cohort_fingerprint"):
        raise AssertionError(
            "Notebook 07 per-study metrics do not match its frozen generation cohort"
        )
    # Label-derived outcomes are deliberately recomputed from the two vectors
    # inside every cluster bootstrap and paired permutation replicate. They are
    # aggregate outcomes, not columns in Notebook 07's per-study JSONL.
    vector_derived_metrics = {
        "fer", "fer_abnormal", "omission", "macro_f1", "micro_f1",
        "hamming_accuracy", "normal_abnormal_accuracy",
        "false_positive_events", "predicted_positive_events",
        "false_negative_events", "reference_positive_events",
        *[f"f1_{label.replace(' ', '_')}" for label in CONFIG["labeler"]["uncertain_policy"]],
    }
    direct_primary_metrics = set(CONFIG["statistics"]["primary_metrics"]) - vector_derived_metrics
    required_columns = {
        "query_record_id", "patient_key", "model_key", "bundle", "source_dataset",
        "condition", "reference_vector", "prediction_vector", *direct_primary_metrics,
    }
    missing_columns = sorted(required_columns - set(frame.columns))
    if missing_columns:
        raise KeyError(f"Per-study metrics are missing statistical inputs: {missing_columns}")
    unavailable_direct_metrics = [
        metric for metric in sorted(direct_primary_metrics)
        if not frame[metric].notna().any()
    ]
    if unavailable_direct_metrics:
        raise RuntimeError(
            "Notebook 07 did not produce usable values for direct primary metrics: "
            f"{unavailable_direct_metrics}. Inspect text_metric_runtime_audit.json before inference."
        )
    expected_conditions = set(CONDITIONS)
    pairing_errors = []
    for keys, group in frame.groupby(
        ["model_key", "bundle", "source_dataset"], dropna=False
    ):
        observed_conditions = set(group["condition"].astype(str))
        if observed_conditions != expected_conditions:
            pairing_errors.append(f"{keys}: conditions={sorted(observed_conditions)}")
            continue
        query_sets = [
            set(group.loc[group["condition"] == condition, "query_record_id"].astype(str))
            for condition in CONDITIONS
        ]
        if any(values != query_sets[0] for values in query_sets[1:]):
            pairing_errors.append(f"{keys}: paired query sets differ across conditions")
    if pairing_errors:
        raise AssertionError(
            "Notebook 08 pairing audit failed: " + "; ".join(pairing_errors[:20])
        )
    statistics_ready = True
    print("NOTEBOOK 08 INPUT AUDIT PASSED:", len(frame), "paired per-study records")

In [ ]:
if not statistics_ready:
    print("Bootstrap intervals skipped because Notebook 07 outputs are pending.")
else:
    reps_b = int(CONFIG["statistics"]["bootstrap_replicates"])
    reps_p = int(CONFIG["statistics"]["permutation_replicates"])
    ci_parts = []
    grouping = ["model_key", "bundle", "source_dataset", "condition"]
    grouped = list(frame.groupby(grouping, dropna=False))
    print(f"Vectorized bootstrap: {len(grouped)} groups × {reps_b:,} patient/source-cluster replicates")
    for index, (keys, group) in enumerate(grouped, start=1):
        ci = cluster_bootstrap(
            group,
            replicates=reps_b,
            confidence_level=CONFIG["statistics"]["confidence_level"],
            seed=CONFIG["statistics"]["seed"],
        )
        for name, value in zip(grouping, keys):
            ci[name] = value
        ci_parts.append(ci)
        if index == 1 or index % 10 == 0 or index == len(grouped):
            print(f"  bootstrap group {index}/{len(grouped)} complete")
    intervals = pd.concat(ci_parts, ignore_index=True)
    intervals_path = PATHS["statistics"] / "cluster_bootstrap_ci.csv"
    intervals.to_csv(intervals_path, index=False)
    print("Saved patient/source-cluster bootstrap intervals:", intervals_path)
    display(intervals.head())

In [ ]:
if not statistics_ready:
    print("Paired tests skipped because Notebook 07 outputs are pending.")
else:
    test_parts = []
    comparisons = [
        ("A_single_pass", "B_unconditional_4pass"),
        ("A_single_pass", "C_pretrained_gate"),
        ("A_single_pass", "D_corrected_lora_gate"),
        ("C_pretrained_gate", "D_corrected_lora_gate"),
    ]
    grouping = ["model_key", "bundle", "source_dataset"]
    grouped = list(frame.groupby(grouping, dropna=False))
    total_within = len(grouped) * len(comparisons)
    print(f"Vectorized within-model tests: {total_within} comparisons × {reps_p:,} paired replicates")
    within_done = 0
    for keys, group in grouped:
        for arm_a, arm_b in comparisons:
            a = group[group["condition"] == arm_a]
            b = group[group["condition"] == arm_b]
            comparison = paired_cluster_permutation(
                a,
                b,
                id_column="query_record_id",
                metrics=CONFIG["statistics"]["primary_metrics"],
                replicates=reps_p,
                seed=CONFIG["statistics"]["seed"],
            )
            comparison["arm_a_condition"] = arm_a
            comparison["arm_b_condition"] = arm_b
            for name, value in zip(grouping, keys):
                comparison[name] = value
            test_parts.append(comparison)
            within_done += 1
            if within_done == 1 or within_done % 25 == 0 or within_done == total_within:
                print(f"  within-model comparison {within_done}/{total_within} complete")
    tests = pd.concat(test_parts, ignore_index=True)
    within_family = ["model_key", "metric", "bundle", "source_dataset"]
    tests["p_holm_within_strategy_family"] = tests.groupby(
        within_family, dropna=False
    )["p_value"].transform(lambda values: holm_adjust(values.tolist()))
    tests_path = PATHS["statistics"] / "paired_cluster_permutation_tests.csv"
    tests.to_csv(tests_path, index=False)

    between_parts = []
    grouping = ["bundle", "source_dataset", "condition"]
    grouped = list(frame.groupby(grouping, dropna=False))
    total_between = sum(len(list(itertools.combinations(sorted(group["model_key"].unique()), 2))) for _, group in grouped)
    print(f"Vectorized between-model tests: {total_between} comparisons × {reps_p:,} paired replicates")
    between_done = 0
    for keys, group in grouped:
        models = sorted(group["model_key"].unique())
        for model_a, model_b in itertools.combinations(models, 2):
            a = group[group["model_key"] == model_a]
            b = group[group["model_key"] == model_b]
            comparison = paired_cluster_permutation(
                a,
                b,
                id_column="query_record_id",
                metrics=CONFIG["statistics"]["primary_metrics"],
                replicates=reps_p,
                seed=CONFIG["statistics"]["seed"],
            )
            comparison["model_a"] = model_a
            comparison["model_b"] = model_b
            for name, value in zip(grouping, keys):
                comparison[name] = value
            between_parts.append(comparison)
            between_done += 1
            if between_done == 1 or between_done % 25 == 0 or between_done == total_between:
                print(f"  between-model comparison {between_done}/{total_between} complete")
    between = pd.concat(between_parts, ignore_index=True)
    between_family = ["metric", "bundle", "source_dataset", "condition"]
    between["p_holm_between_model_family"] = between.groupby(
        between_family, dropna=False
    )["p_value"].transform(lambda values: holm_adjust(values.tolist()))
    between_path = PATHS["statistics"] / "between_model_paired_tests.csv"
    between.to_csv(between_path, index=False)

    write_json(status_path, {
        "ready": True,
        "upstream_status": str(upstream_status_path),
        "upstream_per_study_metrics_sha256": actual_sha256,
        "generation_cohort_fingerprint": actual_fingerprint,
        "n_records": int(len(frame)),
        "bootstrap_replicates": reps_b,
        "permutation_replicates": reps_p,
        "statistics_engine": "vectorized_cluster_sufficient_statistics_v3",
        "confidence_level": CONFIG["statistics"]["confidence_level"],
        "cluster_bootstrap_ci": str(intervals_path),
        "cluster_bootstrap_ci_sha256": sha256_path(intervals_path),
        "paired_cluster_permutation_tests": str(tests_path),
        "paired_cluster_permutation_tests_sha256": sha256_path(tests_path),
        "between_model_paired_tests": str(between_path),
        "between_model_paired_tests_sha256": sha256_path(between_path),
        "within_model_holm_family": within_family,
        "between_model_holm_family": between_family,
    })
    print("NOTEBOOK 08 COMPLETE — CLUSTERED INFERENCE IS LINKED TO NOTEBOOK 07.")
    display(tests.head())
    display(between.head())